# LLM Classification Finetuning

## 1. Train

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch
import evaluate

LLM_MODEL = "distilbert-base-uncased"
MAX_TOKEN_LENGTH = 512
train_data_path = "kaggle/input/competitions/llm-classification-finetuning/train.csv"
test_data_path = "kaggle/input/competitions/llm-classification-finetuning/test.csv"
model_save_path = "./model"
data_submission_path = "submission.csv"


In [ ]:
def maybe_swap(row):
    if np.random.rand() < 0.5:
        row["response_a"], row["response_b"] = row["response_b"], row["response_a"]
        if row["labels"] == 0:
            row["labels"] = 1
        elif row["labels"] == 1:
            row["labels"] = 0
    return row

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_TOKEN_LENGTH,
    )

def truncate_text(text, max_tokens):
    ids = tokenizer(
        str(text),
        truncation=True,
        max_length=max_tokens,
        add_special_tokens=False
    )["input_ids"]
    return tokenizer.decode(ids)

def build_text(row):
    prompt = truncate_text(row["prompt"], 128)
    a = truncate_text(row["response_a"], 192)
    b = truncate_text(row["response_b"], 192)

    return (
        "[PROMPT]\n" + prompt +
        "\n\n[RESPONSE A]\n" + a +
        "\n\n[RESPONSE B]\n" + b
    )

In [ ]:

# Data loading + preprocessing
df = pd.read_csv(train_data_path)

# 1) Label encoding
if set(["winner_model_a", "winner_model_b", "winner_tie"]).issubset(df.columns):
    df["labels"] = (
        df["winner_model_a"] * 0 +
        df["winner_model_b"] * 1 +
        df["winner_tie"] * 2
    ).astype("int64")
else:
    raise ValueError("Missing one of winner_model_a/winner_model_b/winner_tie")

# 2) Ramdomly swap response_a and response_b for half the data to prevent positional bias
df = df.apply(maybe_swap, axis=1)


# 3) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(df.columns):
    df["text"] = df.apply(build_text, axis=1)
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

print("Loaded", len(df), "rows")
print(df["labels"].value_counts())

In [ ]:


# 3) HF dataset
dataset = Dataset.from_pandas(df[["text", "labels"]])

# 4) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]


train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


In [ ]:


model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL,
    num_labels=3
)

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir=model_save_path,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    max_steps=1200,
    warmup_steps=0.1,
    weight_decay=0.01,
    fp16=False,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=120,
    save_steps=480,
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    logging_nan_inf_filter=False,
    # use_cpu=True,
)


In [ ]:


data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics, 
)

In [ ]:
#B. Check a manual forward pass on a train batch

# Do this before full training and then again after a few steps:

name, p = next((n, p) for n, p in model.named_parameters() if p.requires_grad)
before = p.detach().cpu().clone()

trainer.train()

after = dict(model.named_parameters())[name].detach().cpu()
print("param changed?", not torch.equal(before, after))
print("max abs diff:", (after - before).abs().max().item())

In [ ]:
trainer.remove_callback(type(trainer.callback_handler.callbacks[-1]))  # not ideal/generic

In [ ]:


print("global_step:", trainer.state.global_step)
print(trainer.state.log_history[-10:])
print("val metrics:", trainer.evaluate(val_dataset))

pred = trainer.predict(val_dataset)
print("nan/inf", np.any(np.isnan(pred.predictions)), np.any(np.isinf(pred.predictions)))

In [ ]:
# Checking prediction distribution to see if model is learning anything or just predicting one class

pred = trainer.predict(val_dataset)
pred_labels = pred.predictions.argmax(axis=-1)

from collections import Counter
print("pred distribution:", Counter(pred_labels))
print("true distribution:", Counter(val_dataset["labels"]))



In [ ]:

# Inspect token length distribution to see if truncation is happening a lot

lengths = [len(tokenizer(t, truncation=False)["input_ids"]) for t in df["text"].sample(1000, random_state=42)]
print("p50", np.percentile(lengths, 50))
print("p90", np.percentile(lengths, 90))
print("p99", np.percentile(lengths, 99))

# In this case, we see that 90% of the samples are above 256 tokens, which means truncation is happening for most samples. We might want to consider using a longer max_length or a model that can handle longer inputs if we think important information is being truncated.

In [ ]:
# Save the trained model and tokenizer
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)


In [ ]:

# Evaluate and show metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])

## 2.Predict

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(model_save_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_save_path)

model.eval()

In [ ]:
# 1) Load test data
test_df = pd.read_csv(test_data_path)

# 2) Prepare text in same format as training
test_df["text"] = (
    "Prompt:\n" + test_df["prompt"].astype(str) +
    "\n\nResponse A:\n" + test_df["response_a"].astype(str) +
    "\n\nResponse B:\n" + test_df["response_b"].astype(str)
)

test_dataset = Dataset.from_pandas(test_df[["text"]])

test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



In [ ]:


# 4) Predict
preds = trainer.predict(test_dataset)
print("Raw predictions shape:", preds)
logits = preds.predictions  # shape (N, 3)
print("Logits shape:", logits)
logits_tensor = torch.from_numpy(logits).float()

if torch.isnan(logits_tensor).any() or torch.isinf(logits_tensor).any():
    raise ValueError("Logits contain NaN or Inf; check model/prediction pipeline")

# stable softmax to avoid numeric instabilities
logits_stable = logits_tensor - logits_tensor.max(dim=-1, keepdim=True).values
exp_scores = torch.exp(logits_stable)
probs_tensor = exp_scores / exp_scores.sum(dim=-1, keepdim=True)
probs = probs_tensor.cpu().numpy()

# 5) save to submission file
out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv(data_submission_path, index=False)
print(out.head())
